## Unilateral Vocal Fold Paralysis (UVFP) — Quick EDA\n\nThis notebook does a lightweight EDA of the phenotype file:\n`b2ai_adult_dataset/3.0.0/phenotype/diagnosis/unilateral_vocal_fold_paralysis.tsv`\n\nIt assumes you’re using the repo’s `environment.yml` (pandas/numpy installed).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")

DATA_PATH = Path("b2ai_adult_dataset/3.0.0/phenotype/diagnosis/unilateral_vocal_fold_paralysis.tsv")
assert DATA_PATH.exists(), f"Missing file: {DATA_PATH}"

In [ ]:
df = pd.read_csv(DATA_PATH, sep="\t")
df.shape

In [ ]:
df.head(10)

### Basic integrity checks

In [ ]:
checks = {
    "rows": len(df),
    "cols": df.shape[1],
    "participant_id_nulls": int(df["participant_id"].isna().sum()),
    "participant_id_unique": int(df["participant_id"].nunique(dropna=True)),
    "participant_id_duplicates": int(df["participant_id"].duplicated().sum()),
}
checks

### Missingness

In [ ]:
missing_frac = df.isna().mean().sort_values(ascending=False)
missing_frac.head(25)

In [ ]:
plt.figure(figsize=(10, 6))
missing_frac.head(20).iloc[::-1].plot(kind="barh")
plt.title("Top 20 columns by missing fraction")
plt.xlabel("Missing fraction")
plt.tight_layout()

### Key categorical fields

In [ ]:
cat_cols = [
    "diagnosis_uvfp_ds",
    "diagnosis_uvfp_treatment",
    "diagnosis_uvfp_dse",
    "diagnosis_uvfp_etiology",
    "diagnosis_uvfp_gold_standard_diagnosis",
    "diagnosis_uvfp_iatrogenic",
    "diagnosis_uvfp_tumor",
]

cats = {}
for c in cat_cols:
    if c in df.columns:
        cats[c] = df[c].fillna("<NA>").value_counts(dropna=False)

cats

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()

for ax, col in zip(axes, ["diagnosis_uvfp_ds", "diagnosis_uvfp_treatment", "diagnosis_uvfp_dse", "diagnosis_uvfp_etiology"]):
    if col not in df.columns:
        ax.axis("off")
        continue
    vc = df[col].fillna("<NA>").value_counts()
    sns.barplot(x=vc.values, y=vc.index, ax=ax)
    ax.set_title(col)
    ax.set_xlabel("count")
    ax.set_ylabel("")

plt.tight_layout()

### Degree fields (numeric)

In [ ]:
degree_cols = [c for c in df.columns if c.startswith("diagnosis_degree_") and not c.endswith("_2")]
deg = df[degree_cols].apply(pd.to_numeric, errors="coerce")
deg.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=deg, orient="h")
plt.title("Degree fields (boxplots)")
plt.tight_layout()

### Treatment flags (one-hot-ish columns)\n\nThese columns are sparse and typically encode whether a treatment type was selected.

In [ ]:
flag_cols = [c for c in df.columns if (
    "treatment_select___" in c or
    "treatment_surgery___" in c or
    "treatment_surgery_thyroplasty___" in c or
    "treatment_surgery_vfia___" in c
)]

flags = df[flag_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
flag_prevalence = (flags != 0).mean().sort_values(ascending=False)
flag_prevalence.head(30)

### Two small cross-tabs (helpful sanity checks)

In [ ]:
if {"diagnosis_uvfp_etiology", "diagnosis_uvfp_treatment"}.issubset(df.columns):
    display(pd.crosstab(df["diagnosis_uvfp_etiology"], df["diagnosis_uvfp_treatment"], margins=True))

In [ ]:
if {"diagnosis_uvfp_ds", "diagnosis_uvfp_dse"}.issubset(df.columns):
    display(pd.crosstab(df["diagnosis_uvfp_ds"], df["diagnosis_uvfp_dse"], margins=True))